# CryptoLucky — Miniaturas con SDXL (GPU gratis)
Genera miniaturas tematicas vibrantes (estilo slot/casino, originales, sin marcas) para el blog.
**Modelo:** stabilityai/stable-diffusion-xl-base-1.0 — licencia **OpenRAIL++ (uso comercial permitido)**, **ungated** (sin token).
**Coste:** 0€ en Google Colab con GPU T4 gratuita.

(Nota: FLUX.1-schnell esta 'gated' en HuggingFace y requiere token; SDXL es ungated y comercial, por eso se usa aqui.)

## Como usar
1. Menu **Entorno de ejecucion -> Cambiar tipo de entorno -> T4 GPU**.
2. **Entorno de ejecucion -> Ejecutar todo** (Run all).
3. Al final se descarga `thumbs.zip` con un PNG por tema (16:9).

In [ ]:
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf

In [ ]:
import torch
from diffusers import StableDiffusionXLPipeline

# SDXL base 1.0 (OpenRAIL++, comercial, ungated). Cabe en la T4 (~7 GB fp16).
pipe = StableDiffusionXLPipeline.from_pretrained(
    'stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    variant='fp16',
    use_safetensors=True,
)
pipe = pipe.to('cuda')
pipe.watermark = None  # sin watermark (ni invisible)

In [ ]:
import os
os.makedirs('thumbs', exist_ok=True)

BASE = 'premium glossy mobile game tile, vibrant saturated colors, glowing light, sparkles and confetti, 3D render, centered composition, dark vignette edges, high energy'
NEG = 'text, words, letters, watermark, logo, brand name, signature, ugly, blurry, low quality, deformed, extra limbs'

prompts = {
    'casino':   'casino slot machine with golden lucky sevens 777 and bursting gold coins, neon purple and magenta, ' + BASE,
    'mundial':  'golden football trophy next to a shiny soccer ball on a green stadium pitch, emerald green and gold, ' + BASE,
    'bonos':    'golden gift box bursting open with gold coins and cash, orange and red, ' + BASE,
    'staking':  'stack of glowing golden crypto coins rising with an upward chart arrow, gold and amber, ' + BASE,
    'juegos':   'casino dice and poker chips flying, fuchsia pink and magenta, ' + BASE,
    'reviews':  'shiny golden five-star rating badge with glow, gold and yellow, ' + BASE,
    'sinkyc':   'glowing golden shield with a check mark, teal and emerald, ' + BASE,
    'guias':    'open glowing book with golden pages and a lightbulb, indigo blue and gold, ' + BASE,
    'apuestas': 'sports betting target with a golden arrow hitting the center, sky blue and gold, ' + BASE,
}

gen = torch.Generator('cuda').manual_seed(7)
for key, p in prompts.items():
    img = pipe(
        prompt=p,
        negative_prompt=NEG,
        num_inference_steps=30,
        guidance_scale=7.0,
        width=1344,
        height=768,
        generator=gen,
    ).images[0]
    img.save(f'thumbs/{key}.png')
    print('generada:', key)

In [ ]:
import shutil
shutil.make_archive('thumbs', 'zip', 'thumbs')
from google.colab import files
files.download('thumbs.zip')